# Fink/LSST — Flat Light Curves for calibrationnby Block Selection

This notebook retrieves large photometric light curves (detections + forced photometry)  
for objects found in the **LSST Deep Drilling Fields**.  
Objects are classified using **Fink crossmatch metadata** (`f:xm_*` columns).  
The goal: identify which source category produces the **flattest, most stable flux**  
light curves, suitable for atmospheric transparency calibration.

- author : Sylvie Dagoret-Campagne
- last update : 2026-05-08 : add dipole info in conesearch, diaObjects catalogue, diaSources LC, and LC plots

## Revised strategy

1. **Cone-search** each DDF via `api/v1/conesearch` using **`r:` column prefix**  
   → one alert per detection (columns include `r:diaObjectId`, `r:nDiaSources`, crossmatch flags)
2. **Deduplicate** by `diaObjectId`, keep objects with `nDiaSources >= NP_MIN`
3. **Download full light curves** via `api/v1/sources` + `api/v1/fp`
4. **Classify** each object into a group based on Fink crossmatch metadata (`f:xm_*`)
5. **Compute flatness** σ/⟨f⟩ per group and rank for calibration suitability

## Key discovery about the API

- `/api/v1/conesearch` requires `r:` column prefix (not `i:` — that causes 500 error)
- Block flags (`b_*`) are not filterable in API calls — they are boolean columns in
  `api/v1/sources` but classification is best done via `f:xm_*` crossmatch columns
  present in the conesearch results

```
POST https://api.lsst.fink-portal.org/api/v1/conesearch   → alerts in a sky cone (r: prefix)
POST https://api.lsst.fink-portal.org/api/v1/sources      → full diaSources for one object
POST https://api.lsst.fink-portal.org/api/v1/fp           → forced photometry
POST https://api.lsst.fink-portal.org/api/v1/objects      → aggregated object-level statistics
POST https://api.lsst.fink-portal.org/api/v1/blocks       → block flag definitions
POST https://api.lsst.fink-portal.org/api/v1/tags         → Fink classification tags
```

## Available `f:xm_*` crossmatch columns (confirmed from live API)

| Column | Catalogue | Notes |
|--------|-----------|-------|
| `f:xm_gaiadr3_DR3Name` | Gaia DR3 | source name; `nan` if no match |
| `f:xm_gaiadr3_VarFlag` | Gaia DR3 | `'0'` = not variable; other value = variable |
| `f:xm_gaiadr3_Plx` | Gaia DR3 | parallax in mas (null if no measurement) |
| `f:xm_gaiadr3_e_Plx` | Gaia DR3 | parallax uncertainty in mas |
| `f:xm_simbad_otype` | SIMBAD | object type string (null if no match) |
| `f:xm_legacydr8_pstar` | Legacy DR8 | P(star) photometric classifier [0–1] |
| `f:xm_legacydr8_zphot` | Legacy DR3 | photometric redshift |
| `f:xm_legacydr8_e_zphot` | Legacy DR8 | photo-z uncertainty |
| `f:xm_legacydr8_fqual` | Legacy DR8 | photo-z quality flag |
| `f:xm_mangrove_2MASS_name` | Mangrove | 2MASS name of nearby galaxy |
| `f:xm_mangrove_HyperLEDA_name` | Mangrove/HyperLEDA | HyperLEDA galaxy name |
| `f:xm_mangrove_ang_dist` | Mangrove | angular distance to galaxy (arcsec) |
| `f:xm_mangrove_lum_dist` | Mangrove | luminosity distance to galaxy (Mpc) |
| `f:xm_gcvs_type` | GCVS | General Catalogue of Variable Stars type |
| `f:xm_vsx_Type` | VSX | Variable Star Index type string |
| `f:xm_spicy_class` | SPICY | Young Stellar Object (YSO) classification |
| `f:xm_tns_fullname` | TNS | Transient Name Server name |
| `f:xm_tns_type` | TNS | classified type (e.g. SN Ia) |
| `f:xm_tns_redshift` | TNS | spectroscopic redshift |
| `f:xm_x3hsp_type` | 3HSP | High-Synchrotron-Peaked blazar type |
| `f:xm_x4lac_type` | 4LAC | Fermi-LAT 4th AGN catalogue type |
| `f:clf_cats_class` | Fink ML | multi-class classifier (int) |
| `f:clf_cats_score` | Fink ML | multi-class classifier score [0–1] |
| `f:clf_snnSnVsOthers_score` | Fink ML | SN vs Others score [0–1] |
| `r:extendedness` | LSST | morphology: 0=point-source, 1=extended |
| `r:reliability` | LSST | alert reliability score [0–1] |

## Dipole columns (r: prefix, from conesearch and diaSources)

These columns are fetched both in the conesearch (per alert) and in the diaSources download.
They allow identifying dipole detections, which are characteristic of subtraction artefacts
(mis-registered template) rather than genuine astrophysical transients.

| Column | Description |
|--------|-------------|
| `r:isDipole` | Boolean — True if the source was classified as a dipole |
| `r:isNegative` | Boolean — True if the source is a negative dipole lobe |
| `r:dipoleFitAttempted` | Boolean — True if the dipole fit was attempted |
| `r:dipoleFluxDiff` | Flux difference between positive and negative lobes (nJy) |
| `r:dipoleFluxDiffErr` | Uncertainty on `dipoleFluxDiff` |
| `r:dipoleMeanFlux` | Mean of positive and negative lobe fluxes (nJy) |
| `r:dipoleMeanFluxErr` | Uncertainty on `dipoleMeanFlux` |
| `r:dipoleLength` | Angular separation between lobes (arcsec) |
| `r:dipoleAngle` | Position angle of the dipole axis (degrees) |
| `r:dipoleNdata` | Number of pixels used in the dipole fit |
| `r:dipoleChi2` | Chi² of the dipole fit |

In the **diaObject catalogue**, per-object aggregated dipole statistics are stored:
- `dipole_fraction`: fraction of diaSources for that object flagged as dipoles
- `dipole_ndetect`: number of detections flagged as dipoles
- `isDipole_any`: True if at least one detection is a dipole

In the **light curve plots**, detections flagged as `r:isDipole == True` are overlaid
as large light-grey circle markers (∘) on top of the standard psfFlux errorbar.

## Key columns for Rubin DRP (USDF)

| Column | Description | Usage |
|--------|-------------|-------|
| `r:visit` | Visit ID (13-digit int, e.g. `2026030900027`) | Butler `visit=` parameter |
| `r:detector` | CCD detector number (0–188) | Butler `detector=` parameter |
| `r:x`, `r:y` | Pixel position on CCD | Source location on focal plane |
| `r:xErr`, `r:yErr` | Pixel position uncertainties | Source localisation precision |

> `r:visit` format: `AAAAMMJJNNNNN` — e.g. `2026030900027` = 2026-03-09, visit #27.  
> Use directly as `visitId` in Butler to run `isr`, `calibrate`, etc.

## Fink Blocks and Tags

- **Blocks** (`b:*`): boolean flags indicating membership in predefined source groups  
  (e.g. `b:Solar System`, `b:Gaia stars`, etc.) — fetched via `/api/v1/blocks`
- **Tags** (`t:*`): Fink classification tags for transient/variable events  
  (e.g. `SN Ia`, `kilonova`, `AGN`, etc.) — fetched via `/api/v1/tags`
- **Note**: Tags and blocks are NOT used to restrict the object selection.  
  All object types are kept, including extragalactic transients (SNe, AGN, etc.),  
  so that the full diversity of DIA sources is available for downstream analysis.


## 1. Imports & configuration

In [ ]:
import requests
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

print(f"pandas  version : {pd.__version__}")
print(f"numpy   version : {np.__version__}")

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401

    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")

```
| **Name of Field**    | **RA(Degrees)** | **Dec (Degress)**| **Type**          |
| -------------------- | --------------- | ---------------- | ----------------- |
| **Carina**           | 161.5           | -59.7            | Galaxie/Nébuleuse |
| **ELAIS-S1**         | 9.45            | -44.0            | DDF               |
| **COSMOS**           | 150.1           | +2.2             | DDF               |
| **Trifid-Lagoon**    | 270.5           | -23.0            | Nébuleuse         |
| **M49**              | 187.4           | +8.0             | Galaxie           |
| **Rubin_SV_280_-48** | 280.0           | -48.0            | Test SV           |
| **Rubin_SV_320_-15** | 320.0           | -15.0            | Test SV           |
| **Rubin_SV_216_-17** | 216.0           | -17.0            | Test SV           |
| **Rubin_SV_212_-7**  | 212.0           | -7.0             | Test SV           |
| **Rubin_SV_225_-40** | 225.0           | -40.0            | Test SV           |
```

In [ ]:
# ── API base URL ──────────────────────────────────────────────────────────────
FINK_API = "https://api.lsst.fink-portal.org"

# ── User parameters ───────────────────────────────────────────────────────────
NP_MIN = 100  # Minimum nDiaSources per object
NC = 20  # Number of light curves to plot per group
CONE_RADIUS = 1800.0  # Cone search radius in arcsec (0.5 deg per DDF)
# CONE_RADIUS = 3600.0  # Cone search radius in arcsec (1.0 deg per DDF)
BANDS = list("ugrizy")

# Gaia parallax SNR threshold for 'gaia_star_stable_hq' (high-quality standard)
# RPlx = Plx / e_Plx >= GAIA_RPLX_MIN  →  reliable geometric distance
GAIA_RPLX_MIN = 5.0

# LSST Deep Drilling Fields (RA/Dec J2000)
DEEP_FIELDS = {
    "COSMOS": (150.1191, 2.2058),
    "ELAIS-S1": (9.4500, -44.000),
    "XMM-LSS": (35.7080, -4.750),
    "ECDFS": (53.1250, -27.8),
    "EDFS-a": (58.9, -49.315),
    "EDFS-b": (63.6, -47.6),
    "EDFS": (61.24, -48.423),
    "M49": (187.4, 8.0),
}


def classify_object(row: dict) -> tuple[str, str]:
    """
    Classify an object into a calibration group using Fink crossmatch columns.

    All f:xm_* columns confirmed present in the live Fink/LSST API
    (fink-science v8.39, fink-broker v4.1).

    Priority order (first match wins):
      1. solar_system             – SSO flag
      2. tns_transient            – TNS name match (SN, AT, …)
      3. blazar_x3hsp             – 3HSP blazar type
      4. blazar_4lac              – 4LAC Fermi-LAT AGN
      5. spicy_yso                – SPICY YSO class
      6. vsx_variable             – VSX variable star type
      7. gcvs_variable            – GCVS variable star type
      8. gaia_star_variable       – Gaia DR3 VarFlag != "0"
      9. gaia_star_stable_hq      – Gaia, stable, Plx/e_Plx >= GAIA_RPLX_MIN & Plx > 0
     10. gaia_star_stable_unknown – Gaia, stable, poor parallax, G < 19 (bright, likely nearby)
     11. gaia_non_stellar_or_unreliable – Gaia match but quality cuts fail
     12. simbad_variable_*        – SIMBAD known variable types
     13. simbad_star              – SIMBAD stellar types (non-variable)
     14. simbad_galaxy            – SIMBAD galaxy / AGN types
     15. simbad_*                 – any other SIMBAD type
     16. legacy_star              – Legacy DR8 P(star) > 0.8
     17. legacy_galaxy_photo_z    – Legacy DR8 galaxy with reliable photo-z (fqual=1)
     18. legacy_galaxy            – Legacy DR8 P(star) < 0.2
     19. mangrove_hyperleda       – HyperLEDA galaxy match
     20. mangrove_galaxy          – Mangrove 2MASS galaxy match
     21. lsst_extended            – LSST extendedness ~ 1
     22. lsst_pointsource         – LSST extendedness ~ 0
     23. unknown                  – no crossmatch found

    Returns (group_name, target_name) where target_name is the catalogue
    identifier string for the matched source (or None).
    """

    target_name = None

    # ── Helper: safe boolean conversion ──────────────────────────────────────
    # Handles bool, int, and string representations ("True", "1", etc.)
    def to_bool(val) -> bool:
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    # ── Helper: is a string column null / empty / sentinel? ──────────────────
    def is_null(val) -> bool:
        return val is None or str(val).strip() in ("", "None", "nan", "Fail", "null")

    # ── Read relevant columns ─────────────────────────────────────────────────
    simbad = str(row.get("f:xm_simbad_otype", "")).strip()
    gaia_name = str(row.get("f:xm_gaiadr3_DR3Name", "")).strip()
    gaia_var = str(row.get("f:xm_gaiadr3_VarFlag", "")).strip()
    gaia_plx = row.get("f:xm_gaiadr3_Plx", None)
    gaia_eplx = row.get("f:xm_gaiadr3_e_Plx", None)
    legacy = row.get("f:xm_legacydr8_pstar", None)
    leg_zphot = row.get("f:xm_legacydr8_zphot", None)
    leg_fqual = row.get("f:xm_legacydr8_fqual", None)
    mangrove = str(row.get("f:xm_mangrove_2MASS_name", "") or "").strip()
    hypleda = str(row.get("f:xm_mangrove_HyperLEDA_name", "") or "").strip()
    gcvs = str(row.get("f:xm_gcvs_type", "") or "").strip()
    vsx = str(row.get("f:xm_vsx_Type", "") or "").strip()
    spicy = str(row.get("f:xm_spicy_class", "") or "").strip()
    tns_name = str(row.get("f:xm_tns_fullname", "") or "").strip()
    x3hsp = str(row.get("f:xm_x3hsp_type", "") or "").strip()
    x4lac = str(row.get("f:xm_x4lac_type", "") or "").strip()
    # f:is_sso and f:is_cataloged may arrive as bool or as string "True"/"False"
    is_sso = to_bool(row.get("f:is_sso", False))
    is_cat = to_bool(row.get("f:is_cataloged", False))  # reserved for future use
    extend = row.get("r:extendedness", None)

    # ── 1. Solar system objects ───────────────────────────────────────────────
    if is_sso:
        return "solar_system", target_name

    # ── 2. Known transients from TNS (SN, AT, …) ─────────────────────────────
    # Kept in the catalogue — classified but NOT excluded from the visit index.
    if not is_null(tns_name):
        return "tns_transient", tns_name

    # ── 3. High-energy blazars (3HSP / 4LAC) ──────────────────────────────────
    if not is_null(x3hsp):
        return "blazar_x3hsp", x3hsp
    if not is_null(x4lac):
        return "blazar_4lac", x4lac

    # ── 4. Young Stellar Objects from SPICY ───────────────────────────────────
    if not is_null(spicy):
        return "spicy_yso", spicy

    # ── 5. Variable stars: VSX then GCVS ──────────────────────────────────────
    if not is_null(vsx):
        return "vsx_variable", vsx
    if not is_null(gcvs):
        return "gcvs_variable", gcvs

    # ── 6. Gaia DR3 stars ─────────────────────────────────────────────────────
    if not is_null(gaia_name):
        # Check variability flag.
        # Doc: "0" = not variable; any other non-null value = variable.
        # We treat "0" and empty/null as stable; everything else as variable.
        is_gaia_var = not is_null(gaia_var) and gaia_var not in ("0", "False", "NOT_AVAILABLE")
        if is_gaia_var:
            return "gaia_star_variable", gaia_name

        # Compute parallax S/N to select high-quality astrometric standards.
        try:
            plx = float(gaia_plx) if gaia_plx is not None else np.nan
            eplx = float(gaia_eplx) if gaia_eplx is not None else np.nan
            rplx = plx / eplx if (np.isfinite(plx) and np.isfinite(eplx) and eplx > 0) else np.nan

            # High-quality parallax: S/N >= threshold AND positive parallax
            parallax_ok = np.isfinite(rplx) and rplx >= GAIA_RPLX_MIN and plx > 0

            if parallax_ok:
                return "gaia_star_stable_hq", gaia_name

            # Poor parallax: fall back on G-band magnitude as a brightness proxy.
            # Bright stars (G < 19) are likely nearby Milky Way members —
            # kept as potential calibrators even without a reliable distance.
            raw_mag = row.get("f:xm_gaiadr3_PhotGMag", None)
            phot_g_mean_mag = float(raw_mag) if raw_mag is not None else None

            if phot_g_mean_mag is not None and phot_g_mean_mag < 19:
                return "gaia_brightstar_stable_unknown_parallax", gaia_name
            elif phot_g_mean_mag is not None and phot_g_mean_mag >= 19:
                return "gaia_faintstar_stable_unknown_parallax", gaia_name
            elif phot_g_mean_mag is None:
                return "gaia_nophotgstar_stable_unknown_parallax", gaia_name
            else:
                return "gaia_unknown_stable_unknown_parallax", gaia_name

        except (TypeError, ValueError):
            # Parallax or magnitude columns are non-numeric — cannot assess quality.
            return "gaia_unreliable", gaia_name

    # ── 7. SIMBAD classifications ──────────────────────────────────────────────
    galaxy_types = {
        "G",
        "GiG",
        "GiC",
        "GiP",
        "GiR",
        "GiS",
        "GiPair",
        "Galaxy",
        "AGN",
        "QSO",
        "Sy1",
        "Sy2",
        "BLL",
        "BlL",
        "GinCl",
        "GinGroup",
        "BClG",
        "ClG",
        "PairG",
    }
    star_types = {
        "*",
        "**",
        "SB*",
        "PM*",
        "V*",
        "RGB*",
        "LP*",
        "Ev*",
        "Em*",
        "WR*",
        "Be*",
        "BS*",
        "S*b",
        "HB*",
        "sg*",
        "Cep",
        "RRLyr",
        "EB*",
        "Mira",
        "LPV*",
        "SRS",
        "RotV*",
        "EllipV*",
        "RS*",
        "BY*",
        "Fl*",
        "YSO",
        "TTau*",
        "HerbObj",
        "pMS*",
        "Ae*",
        "bCep",
        "SX*",
        "dS*",
        "WVir",
        "RV*",
        "deltaCep",
        "gammaDor",
        "roAp",
    }
    # Subset of star_types that are intrinsically variable —
    # tested first so they are not silently folded into "simbad_star".
    simbad_variable_types = {
        "Cep",
        "RRLyr",
        "EB*",
        "Mira",
        "LPV*",
        "SRS",
        "RotV*",
        "EllipV*",
        "RS*",
        "BY*",
        "Fl*",
        "TTau*",
        "bCep",
        "SX*",
        "dS*",
        "WVir",
        "RV*",
        "deltaCep",
        "gammaDor",
        "roAp",
        "V*",
        "YSO",
        "HerbObj",
        "pMS*",
        "Ae*",
    }

    if not is_null(simbad):
        if simbad in simbad_variable_types:
            return f"simbad_variable_{simbad}", simbad
        if simbad in star_types:
            return "simbad_star", simbad
        if simbad in galaxy_types:
            return "simbad_galaxy", simbad
        return f"simbad_{simbad}", simbad

    # ── 8. Legacy DR8 photometric star / galaxy separator ─────────────────────
    if legacy is not None and not isinstance(legacy, str):
        try:
            p = float(legacy)
            if p > 0.8:
                return "legacy_star", target_name
            if p < 0.2:
                # Check whether a reliable photometric redshift is available.
                try:
                    fq = int(leg_fqual)
                    zp = float(leg_zphot)
                    if fq == 1 and np.isfinite(zp):
                        return "legacy_galaxy_photo_z", target_name
                except (TypeError, ValueError):
                    pass
                return "legacy_galaxy", target_name
        except (TypeError, ValueError):
            pass

    # ── 9. Mangrove / HyperLEDA galaxy environment ────────────────────────────
    if not is_null(hypleda):
        return "mangrove_hyperleda", hypleda
    if not is_null(mangrove):
        return "mangrove_galaxy", mangrove

    # ── 10. LSST morphology (last resort before unknown) ──────────────────────
    if extend is not None:
        try:
            e = float(extend)
            if e >= 0.5:
                return "lsst_extended", target_name
            return "lsst_pointsource", target_name
        except (TypeError, ValueError):
            pass

    return "unknown", target_name


# ── Classification groups based on Fink crossmatch metadata ──────────────────
# Priority order matters: first match wins.
#
# NOTE: All object types are kept including transients (SNe, AGN, etc.).
# Tags and blocks are retrieved for information only, NOT used to filter objects.
# Groups added in v2 compared to v1:
#   - gaia_star_stable_hq  : Gaia stable + parallax SNR >= GAIA_RPLX_MIN (best calibrators)
#   - vsx_variable         : Variable Star Index match (flag explicitly)
#   - gcvs_variable        : GCVS match (similar to VSX)
#   - spicy_yso            : Young Stellar Object from SPICY
#   - tns_transient        : Transient Name Server match (SN, AT, etc.)
#   - blazar_x3hsp         : High-Synchrotron-Peaked blazar (3HSP)
#   - blazar_4lac          : Fermi-LAT AGN (4LAC)
#   - mangrove_hyperleda   : HyperLEDA galaxy
#   - legacy_galaxy_photo_z: Legacy galaxy with reliable photo-z (fqual=1)
#   - lsst_extended        : morphologically extended source (r:extendedness ~ 1)
#   - lsst_pointsource     : morphologically compact (r:extendedness ~ 0)


# ── Output directories ────────────────────────────────────────────────────────
NB_TAG = "FINK_BLOCK_LC_01_test"
DIR_DATA = f"data_{NB_TAG}"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data : {os.path.abspath(DIR_DATA)}")
print(f"Figs : {os.path.abspath(DIR_FIGS)}")

BAND_COLORS = {
    "u": "#9b59b6",
    "g": "#2ecc71",
    "r": "#e74c3c",
    "i": "#e67e22",
    "z": "#3498db",
    "y": "#795548",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. API wrappers

In [ ]:
def _post_json(url, payload, timeout=90):
    """POST a JSON payload and return the parsed response, raising on HTTP errors."""
    r = requests.post(url, json=payload, timeout=timeout)
    r.raise_for_status()
    return r.json()


def fetch_conesearch(
    ra: float, dec: float, radius: float, n: int = 1000, columns: str | None = None
) -> pd.DataFrame:
    """
    Cone search via /api/v1/conesearch.
    IMPORTANT: column names must use 'r:' or 'f:' prefix — NOT 'i:'.
    Returns one row per alert/diaSource (not per object).
    """
    payload = {
        "ra": ra,
        "dec": dec,
        "radius": radius,
        "n": n,
        "output-format": "json",
    }
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/conesearch", payload)
    if not raw:
        return pd.DataFrame()
    return pd.DataFrame(raw)


def fetch_sources(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch diaSources (direct detections) for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/sources", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_fp(diaObjectId, columns=None) -> pd.DataFrame:
    """Fetch forced photometry for one diaObjectId."""
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/fp", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects(diaObjectId, columns=None) -> pd.DataFrame:
    """
    Fetch aggregated object-level statistics for one diaObjectId via /api/v1/objects.

    The 'i:' prefix is used by this endpoint (unlike 'r:' for conesearch/sources).
    When columns=None, ALL available columns are returned (recommended for catalogues).

    Key aggregated columns include (per-band prefix u/g/r/i/z/y):
      Global : r:ra, r:dec, r:nDiaSources, r:firstDiaSourceMjdTai, r:lastDiaSourceMjdTai,
               r:target_name, r:observationReason, r:radecMjdTai
      Per-band fluxes (psfFlux):
               r:{b}_psfFluxMean, r:{b}_psfFluxMeanErr, r:{b}_psfFluxSigma,
               r:{b}_psfFluxMAD, r:{b}_psfFluxMax, r:{b}_psfFluxMin, r:{b}_psfFluxNData,
               r:{b}_psfFluxLinearSlope, r:{b}_psfFluxLinearIntercept, r:{b}_psfFluxChi2,
               r:{b}_psfFluxPercentile05/25/50/75/95
      Per-band science flux:
               r:{b}_scienceFluxMean, r:{b}_scienceFluxMeanErr
      Per-band template flux:
               r:{b}_templateFluxMean, r:{b}_templateFluxMeanErr
      Per-band apFlux:
               r:{b}_apFluxMean, r:{b}_apFluxMeanErr, r:{b}_apFluxSigma
    """
    payload = {"diaObjectId": str(diaObjectId), "output-format": "json"}
    if columns:
        payload["columns"] = columns
    raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
    return pd.DataFrame(raw) if raw else pd.DataFrame()


def fetch_objects_batch(diaObjectIds: list, columns=None) -> pd.DataFrame:
    """
    Fetch aggregated object-level statistics for a list of diaObjectIds in a
    single API call (comma-separated IDs).

    The /api/v1/objects endpoint accepts a comma-separated list of IDs,
    which is much more efficient than one call per object.

    Parameters
    ----------
    diaObjectIds : list of str or int
    columns : str or None
        Comma-separated column names (i: prefix).  None = return all columns.

    Returns
    -------
    pd.DataFrame  — one row per diaObjectId, or empty DataFrame on failure.
    """
    ids_str = ",".join(str(oid) for oid in diaObjectIds)
    payload = {"diaObjectId": ids_str, "output-format": "json"}
    if columns:
        payload["columns"] = columns
    try:
        raw = _post_json(f"{FINK_API}/api/v1/objects", payload)
        return pd.DataFrame(raw) if raw else pd.DataFrame()
    except Exception as e:
        print(f"fetch_objects_batch ERROR for {len(diaObjectIds)} IDs: {e}")
        return pd.DataFrame()


def fetch_blocks() -> pd.DataFrame:
    """
    Fetch the Fink block definitions from /api/v1/blocks.

    The Fink LSST API uses POST for all endpoints, including those that
    take no mandatory parameters. The 'output-format': 'json' key is
    required to ensure a JSON response rather than CSV or Avro.

    The response may be:
      - a list of dicts  → pd.DataFrame(data)
      - a list of strings / scalars  → pd.DataFrame({'name': data})
      - a plain dict  → pd.DataFrame([data])

    Returns an empty DataFrame if the endpoint is unavailable or returns
    no data, with a descriptive message for each failure mode.
    """
    url = f"{FINK_API}/api/v1/blocks"
    try:
        r = requests.get(url)
        if r.status_code in (404, 405):
            print(f"fetch_blocks: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            print("fetch_blocks: empty response from API")
            return pd.DataFrame()
        if isinstance(data, list):
            if len(data) == 0:
                return pd.DataFrame()
            if isinstance(data[0], dict):
                return pd.DataFrame(data)
            else:
                return pd.DataFrame({"name": data})
        if isinstance(data, dict):
            return pd.DataFrame([data])
        return pd.DataFrame({"raw": [data]})
    except requests.exceptions.HTTPError as e:
        print(f"fetch_blocks HTTP error: {e}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as e:
        print(f"fetch_blocks connection error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"fetch_blocks unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()


def fetch_tags() -> pd.DataFrame:
    """
    Fetch the Fink classification tags from /api/v1/tags.

    Tags represent event-level classifications (SN Ia, kilonova, AGN flare, etc.).
    NOTE: tags are retrieved for information only — objects are NOT filtered by tag.
    """
    url = f"{FINK_API}/api/v1/tags"
    try:
        r = requests.get(url)
        if r.status_code in (404, 405):
            print(f"fetch_tags: endpoint not available (HTTP {r.status_code})")
            return pd.DataFrame()
        r.raise_for_status()
        data = r.json()
        if not data:
            print("fetch_tags: empty response from API")
            return pd.DataFrame()
        if isinstance(data, list):
            if len(data) == 0:
                return pd.DataFrame()
            if isinstance(data[0], dict):
                return pd.DataFrame(data)
            else:
                return pd.DataFrame({"name": data})
        if isinstance(data, dict):
            return pd.DataFrame([data])
        return pd.DataFrame({"raw": [data]})
    except requests.exceptions.HTTPError as e:
        print(f"fetch_tags HTTP error: {e}")
        return pd.DataFrame()
    except requests.exceptions.ConnectionError as e:
        print(f"fetch_tags connection error: {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"fetch_tags unexpected error: {type(e).__name__}: {e}")
        return pd.DataFrame()


print("API helpers defined.")

## 2b. Fetch Fink Blocks and Tags (informational)

In [ ]:
# Fetch available blocks from Fink API — informational only, NOT used for filtering
df_blocks = fetch_blocks()
if not df_blocks.empty:
    print(f"Fink Blocks ({len(df_blocks)} entries):")
    print(df_blocks.to_string(index=False))
    display(df_blocks.T)
else:
    print("No blocks returned (endpoint may not be available in this API version).")

In [ ]:
# Fetch available tags from Fink API — informational only, NOT used for filtering.
df_tags = fetch_tags()
if not df_tags.empty:
    print(f"Fink Tags ({len(df_tags)} entries):")
    print(df_tags.to_string(index=False))
    display(df_tags.T)
else:
    print("No tags returned (endpoint may not be available in this API version).")

## 3. Utility functions

In [ ]:
AB_FLUX_ZERO = 3631e9  # nJy at AB zero-point


def flux_to_mag(flux_nJy, flux_err_nJy=None):
    flux = np.asarray(flux_nJy, dtype=float)
    with np.errstate(invalid="ignore", divide="ignore"):
        mag = np.where(flux > 0, -2.5 * np.log10(flux / AB_FLUX_ZERO), np.nan)
    mag_err = None
    if flux_err_nJy is not None:
        err = np.asarray(flux_err_nJy, dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            mag_err = np.where(flux > 0, 2.5 / np.log(10) * np.abs(err / flux), np.nan)
    return mag, mag_err


def rms_variability(flux):
    f = np.asarray(flux, dtype=float)
    f = f[np.isfinite(f) & (f > 0)]
    return float(np.std(f) / np.mean(f)) if len(f) >= 3 else np.nan


def filter_lc(
    df_lc,
    mjd_col="r:midpointMjdTai",
    flux_col="r:psfFlux",
    ferr_col="r:psfFluxErr",
    band_col="r:band",
    snr_min=3.0,
):
    """
    Filter a raw light-curve DataFrame: drop low-SNR points, convert to magnitudes.
    Works for both diaSources and forced photometry DataFrames.
    Dipole columns (r:isDipole, r:isNegative, r:dipoleFitAttempted, etc.) are
    preserved if present.
    """
    df = df_lc.copy()
    for col in (mjd_col, flux_col, ferr_col, band_col):
        if col not in df.columns:
            return pd.DataFrame()
    df[flux_col] = pd.to_numeric(df[flux_col], errors="coerce")
    df[ferr_col] = pd.to_numeric(df[ferr_col], errors="coerce")
    df[mjd_col] = pd.to_numeric(df[mjd_col], errors="coerce")
    snr = df[flux_col].abs() / df[ferr_col].replace(0, np.nan)
    df = df[snr >= snr_min].sort_values(mjd_col).reset_index(drop=True)
    df = df.dropna(subset=[flux_col, ferr_col, mjd_col]).reset_index(drop=True)
    mag, mag_err = flux_to_mag(df[flux_col].values, df[ferr_col].values)
    df["mag"] = mag
    df["mag_err"] = mag_err
    df = df.dropna(subset=["mag", "mag_err"]).reset_index(drop=True)
    return df


def parse_dipole_bool(series: pd.Series) -> pd.Series:
    """
    Convert a dipole boolean column (may arrive as bool, int, or string) to bool.
    Returns a boolean Series aligned with the input index.
    """

    def _to_bool(val):
        if isinstance(val, bool):
            return val
        if isinstance(val, (int, float)):
            return bool(val)
        if isinstance(val, str):
            return val.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_to_bool)


print("Utility functions defined.")

## 4. Cone searches on Deep Drilling Fields

**Key fix**: column names in conesearch must use `r:` prefix (not `i:`).  
The `r:nDiaSources` column gives the total detection count per object.

**v4 additions**:
- `r:visit`, `r:detector`: required for Butler-based DRP reconstruction at USDF
- `r:x`, `r:y`: pixel position on the CCD
- `r:xErr`, `r:yErr`: pixel position uncertainties

**v5 additions**:
- `f:tags`: Fink classification tags fetched alongside crossmatch info
  (no filtering applied — all tagged objects including SNe are kept)

**v6 additions**:
- Dipole columns: `r:isDipole`, `r:isNegative`, `r:dipoleFitAttempted`,
  `r:dipoleFluxDiff`, `r:dipoleFluxDiffErr`, `r:dipoleMeanFlux`, `r:dipoleMeanFluxErr`,
  `r:dipoleLength`, `r:dipoleAngle`, `r:dipoleNdata`, `r:dipoleChi2`.
  These columns are fetched in the conesearch AND in the diaSources download.
  Per-object dipole statistics (fraction, count, any-flag) are stored in `all_candidates`
  and propagated to the diaObject catalogue.

**Aggregation strategy (v3 fix)**: `f:xm_*` columns can be null on some alerts
of the same object (e.g. `f:xm_vsx_Type` alternates between 'RRAB' and 'nan'
for a confirmed RR Lyrae). Fix: aggregate all alerts per object and take the
**mode of non-null values** before classifying.

In [ ]:
# ── Columns to fetch from conesearch ─────────────────────────────────────────
# All confirmed from live Fink/LSST API.
# NOTE: r:xErr and r:yErr may not be available in all API versions;
#       they are included here and handled gracefully if absent.
# Dipole columns (v6): fetched per-alert and aggregated per object.
COLS_CONE = (
    "r:diaObjectId,r:nDiaSources,r:ra,r:dec,r:band,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:extendedness,"
    "r:target_name,"
    "r:isNegative,"
    "r:isDipole,"
    "r:dipoleFluxDiff,"
    "r:dipoleFluxDiffErr,"
    "r:dipoleMeanFlux,"
    "r:dipoleLength,"
    "r:dipoleAngle,"
    "r:dipoleMeanFluxErr,"
    "r:dipoleNdata,"
    "r:dipoleChi2,"
    "r:dipoleFitAttempted,"
    # Rubin DRP columns: visit, detector, pixel position + uncertainties
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    # Gaia DR3 (name, variability flag, parallax + uncertainty)
    "f:xm_gaiadr3_DR3Name,f:xm_gaiadr3_VarFlag,"
    "f:xm_gaiadr3_Plx,f:xm_gaiadr3_e_Plx,"
    # SIMBAD
    "f:xm_simbad_otype,"
    # Legacy DR8 (star/galaxy separator + photo-z)
    "f:xm_legacydr8_pstar,f:xm_legacydr8_zphot,f:xm_legacydr8_fqual,"
    # Mangrove (nearby galaxies: 2MASS + HyperLEDA)
    "f:xm_mangrove_2MASS_name,f:xm_mangrove_HyperLEDA_name,"
    # Variable star catalogues (VSX + GCVS)
    "f:xm_vsx_Type,f:xm_gcvs_type,"
    # Young Stellar Objects (SPICY)
    "f:xm_spicy_class,"
    # Transient Name Server
    "f:xm_tns_fullname,f:xm_tns_type,"
    # High-energy blazars (3HSP + 4LAC)
    "f:xm_x3hsp_type,f:xm_x4lac_type,"
    # Fink flags
    "f:is_sso,f:is_cataloged,"
    # Classes and score
    "f:clf_cats_class,f:clf_cats_score,f:clf_earlySNIa_score,f:clf_elephant_kstest_science,f:clf_elephant_kstest_template,"
    "f:clf_slsn_score,f:clf_snnSnVsOthers_score,f:main_label_classifier,f:main_label_crossmatch,"
)

# ── Dipole columns present in conesearch (per-alert, aggregated per object) ──
DIPOLE_COLS_CONE = [
    "r:isDipole",
    "r:isNegative",
    "r:dipoleFitAttempted",
    "r:dipoleFluxDiff",
    "r:dipoleFluxDiffErr",
    "r:dipoleMeanFlux",
    "r:dipoleMeanFluxErr",
    "r:dipoleLength",
    "r:dipoleAngle",
    "r:dipoleNdata",
    "r:dipoleChi2",
]

# Crossmatch columns subject to alert-level inconsistency (see v3 note above).
XM_COLS = [
    "f:xm_gaiadr3_DR3Name",
    "f:xm_gaiadr3_VarFlag",
    "f:xm_gaiadr3_Plx",
    "f:xm_gaiadr3_e_Plx",
    "f:xm_simbad_otype",
    "f:xm_legacydr8_pstar",
    "f:xm_legacydr8_zphot",
    "f:xm_legacydr8_fqual",
    "f:xm_mangrove_2MASS_name",
    "f:xm_mangrove_HyperLEDA_name",
    "f:xm_vsx_Type",
    "f:xm_gcvs_type",
    "f:xm_spicy_class",
    "f:xm_tns_fullname",
    "f:xm_tns_type",
    "f:xm_x3hsp_type",
    "f:xm_x4lac_type",
]
# Boolean Fink flags — treated with majority vote, NOT string mode.
# CRITICAL: bool('False') == True in Python!
BOOL_COLS = ["f:is_sso", "f:is_cataloged"]

NULL_VALS = {"", "None", "nan", "Fail", "null", "NaN"}


def aggregate_xm_for_object(df_alerts):
    """
    Given all conesearch alerts for one object, return a single representative
    dict of crossmatch values by taking, for each XM_COLS column, the MODE of
    all non-null values across the alerts.

    Also computes per-object dipole statistics:
      - dipole_fraction   : fraction of alerts flagged as r:isDipole
      - dipole_ndetect    : number of alerts flagged as r:isDipole
      - isDipole_any      : True if at least one alert is a dipole
      - dipoleFluxDiff_median   : median r:dipoleFluxDiff over dipole detections
      - dipoleMeanFlux_median   : median r:dipoleMeanFlux over dipole detections
      - dipoleLength_median     : median r:dipoleLength over dipole detections
      - dipoleAngle_median      : median r:dipoleAngle over dipole detections
      - dipoleChi2_median       : median r:dipoleChi2 over dipole detections
    """
    best = {}
    for col in XM_COLS:
        if col not in df_alerts.columns:
            best[col] = None
            continue
        vals = df_alerts[col].dropna().astype(str)
        vals = vals[~vals.isin(NULL_VALS)]
        best[col] = vals.mode().iloc[0] if not vals.empty else None
    # Boolean Fink flags: majority vote over all alerts.
    for col in BOOL_COLS:
        if col not in df_alerts.columns:
            best[col] = False
            continue
        bvals = df_alerts[col].dropna()
        if bvals.empty:
            best[col] = False
        else:
            bool_series = bvals.apply(lambda v: str(v).strip().lower() in ("true", "1", "yes"))
            best[col] = bool(bool_series.mean() > 0.5)

    # ── Per-object dipole statistics (aggregated over all alerts) ─────────────
    if "r:isDipole" in df_alerts.columns:
        is_dipole_series = parse_dipole_bool(df_alerts["r:isDipole"].fillna(False))
        n_total = len(df_alerts)
        n_dipole = int(is_dipole_series.sum())
        best["dipole_ndetect"] = n_dipole
        best["dipole_fraction"] = n_dipole / n_total if n_total > 0 else 0.0
        best["isDipole_any"] = n_dipole > 0
        # Aggregate numerical dipole columns over dipole-flagged rows only
        df_dip = df_alerts[is_dipole_series]
        for src_col, dst_key in [
            ("r:dipoleFluxDiff", "dipoleFluxDiff_median"),
            ("r:dipoleMeanFlux", "dipoleMeanFlux_median"),
            ("r:dipoleLength", "dipoleLength_median"),
            ("r:dipoleAngle", "dipoleAngle_median"),
            ("r:dipoleChi2", "dipoleChi2_median"),
        ]:
            if src_col in df_dip.columns and not df_dip.empty:
                vals = pd.to_numeric(df_dip[src_col], errors="coerce").dropna()
                best[dst_key] = float(vals.median()) if not vals.empty else float("nan")
            else:
                best[dst_key] = float("nan")
    else:
        best["dipole_ndetect"] = 0
        best["dipole_fraction"] = 0.0
        best["isDipole_any"] = False
        for k in (
            "dipoleFluxDiff_median",
            "dipoleMeanFlux_median",
            "dipoleLength_median",
            "dipoleAngle_median",
            "dipoleChi2_median",
        ):
            best[k] = float("nan")

    return best


all_candidates = {}  # diaObjectId (str) -> metadata dict

# loop on all Deep field and make a cone search for the alerts
for field_name, (ra, dec) in DEEP_FIELDS.items():
    print(f'\n── Cone search: {field_name}  RA={ra:.4f}  Dec={dec:.4f}  r={CONE_RADIUS:.0f}"')
    try:
        df_cone = fetch_conesearch(ra, dec, CONE_RADIUS, n=2000, columns=COLS_CONE)
    except Exception as e:
        print(f"  ERROR: {e}")
        continue

    if df_cone.empty:
        print("  No alerts returned.")
        continue

    print(f"  Alerts returned: {len(df_cone)}")

    oid_col = "r:diaObjectId"
    nsrc_col = "r:nDiaSources"

    if oid_col not in df_cone.columns:
        print(f"  Column {oid_col!r} missing — cols: {df_cone.columns.tolist()[:10]}")
        continue

    if nsrc_col in df_cone.columns:
        df_cone[nsrc_col] = pd.to_numeric(df_cone[nsrc_col], errors="coerce").fillna(0)

    # ── Aggregate crossmatch columns per object (mode over all alerts) ────────
    grouped = df_cone.groupby(oid_col)
    df_nsrc = df_cone.groupby(oid_col)[nsrc_col].max().reset_index() if nsrc_col in df_cone.columns else None
    df_pos = df_cone.groupby(oid_col)[["r:ra", "r:dec"]].first().reset_index()

    aggregated_rows = []
    for oid, grp in grouped:
        row = {oid_col: oid}
        row.update(aggregate_xm_for_object(grp))
        aggregated_rows.append(row)
    df_obj = pd.DataFrame(aggregated_rows)
    if df_nsrc is not None:
        df_obj = df_obj.merge(df_nsrc, on=oid_col, how="left")
    df_obj = df_obj.merge(df_pos, on=oid_col, how="left")

    print(f"  Unique objects: {len(df_obj)}")

    # select a subset of objects with the requested number of diasources
    df_ok = df_obj[df_obj[nsrc_col] >= NP_MIN] if nsrc_col in df_obj.columns else df_obj
    print(f"  Objects with >={NP_MIN} detections: {len(df_ok)}")

    for _, row in df_ok.iterrows():
        oid = str(row[oid_col])
        nsrc = int(row.get(nsrc_col, 0))
        group, target_name = classify_object(row.to_dict())
        if oid not in all_candidates:
            all_candidates[oid] = {
                "nDiaSources": nsrc,
                "group": group,
                "field": field_name,
                "ra": float(row.get("r:ra", float("nan"))),
                "dec": float(row.get("r:dec", float("nan"))),
                "target_name": target_name,
                # Gaia DR3 crossmatch columns (already aggregated per object)
                "xm_gaiadr3_DR3Name": row.get("f:xm_gaiadr3_DR3Name"),
                "xm_gaiadr3_VarFlag": row.get("f:xm_gaiadr3_VarFlag"),
                "xm_gaiadr3_Plx": row.get("f:xm_gaiadr3_Plx"),
                "xm_gaiadr3_e_Plx": row.get("f:xm_gaiadr3_e_Plx"),
                # SIMBAD
                "xm_simbad_otype": row.get("f:xm_simbad_otype"),
                # Variable star catalogues
                "xm_vsx_Type": row.get("f:xm_vsx_Type"),
                "xm_gcvs_type": row.get("f:xm_gcvs_type"),
                # TNS (transients)
                "xm_tns_fullname": row.get("f:xm_tns_fullname"),
                "xm_tns_type": row.get("f:xm_tns_type"),
                # Legacy DR8
                "xm_legacydr8_pstar": row.get("f:xm_legacydr8_pstar"),
                # Fink flags
                "is_sso": row.get("f:is_sso"),
                "is_cataloged": row.get("f:is_cataloged"),
                # ── Dipole statistics (aggregated from conesearch alerts) ──────
                "dipole_ndetect": row.get("dipole_ndetect", 0),
                "dipole_fraction": row.get("dipole_fraction", 0.0),
                "isDipole_any": row.get("isDipole_any", False),
                "dipoleFluxDiff_median": row.get("dipoleFluxDiff_median", float("nan")),
                "dipoleMeanFlux_median": row.get("dipoleMeanFlux_median", float("nan")),
                "dipoleLength_median": row.get("dipoleLength_median", float("nan")),
                "dipoleAngle_median": row.get("dipoleAngle_median", float("nan")),
                "dipoleChi2_median": row.get("dipoleChi2_median", float("nan")),
            }

    time.sleep(0.5)

print(f"\n=== Total unique candidates: {len(all_candidates)} ===")

import collections

group_counts = collections.Counter(v["group"] for v in all_candidates.values())
print("\nGroup distribution:")
for g, n in sorted(group_counts.items(), key=lambda x: -x[1]):
    print(f"  {g:45s} {n:4d} objects")

n_any_dipole = sum(1 for v in all_candidates.values() if v.get("isDipole_any", False))
print(f"\nObjects with at least one dipole detection: {n_any_dipole} / {len(all_candidates)}")

## 4b. Build base diaObject catalogue (from conesearch)

Build a **flat catalogue** of all diaObjects with one row per object from
the conesearch data already in memory.  This provides `ra`, `dec`,
`target_name`, classification metadata, Gaia DR3 crossmatch columns,
and the **per-object dipole statistics** aggregated in section 4.

The catalogue is **enriched in section 4c** with all aggregated flux statistics
retrieved from `/api/v1/objects`.

In [ ]:
all_candidates["170019716302635146"]

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Build the base diaObject catalogue from all_candidates (conesearch data)
# ──────────────────────────────────────────────────────────────────────────────

catalogue_rows = []
for oid, meta in all_candidates.items():
    row = {
        "diaObjectId": oid,
        "group": meta.get("group"),
        "target_name": meta.get("target_name"),
        "ra": meta.get("ra"),
        "dec": meta.get("dec"),
        "field": meta.get("field"),
        "nDiaSources": meta.get("nDiaSources"),
        "xm_gaiadr3_DR3Name": meta.get("xm_gaiadr3_DR3Name"),
        "xm_gaiadr3_VarFlag": meta.get("xm_gaiadr3_VarFlag"),
        "xm_gaiadr3_Plx": meta.get("xm_gaiadr3_Plx"),
        "xm_gaiadr3_e_Plx": meta.get("xm_gaiadr3_e_Plx"),
        "xm_simbad_otype": meta.get("xm_simbad_otype"),
        "xm_vsx_Type": meta.get("xm_vsx_Type"),
        "xm_gcvs_type": meta.get("xm_gcvs_type"),
        "xm_tns_fullname": meta.get("xm_tns_fullname"),
        "xm_tns_type": meta.get("xm_tns_type"),
        "xm_legacydr8_pstar": meta.get("xm_legacydr8_pstar"),
        "is_sso": meta.get("is_sso"),
        "is_cataloged": meta.get("is_cataloged"),
        # ── Dipole statistics (aggregated from conesearch alerts) ──────────────
        "dipole_ndetect": meta.get("dipole_ndetect", 0),
        "dipole_fraction": meta.get("dipole_fraction", 0.0),
        "isDipole_any": meta.get("isDipole_any", False),
        "dipoleFluxDiff_median": meta.get("dipoleFluxDiff_median", float("nan")),
        "dipoleMeanFlux_median": meta.get("dipoleMeanFlux_median", float("nan")),
        "dipoleLength_median": meta.get("dipoleLength_median", float("nan")),
        "dipoleAngle_median": meta.get("dipoleAngle_median", float("nan")),
        "dipoleChi2_median": meta.get("dipoleChi2_median", float("nan")),
    }
    catalogue_rows.append(row)

df_diaobj_catalogue = pd.DataFrame(catalogue_rows)


# Compute Gaia parallax SNR for convenience
def _safe_rplx(row):
    try:
        plx = (
            float(row["xm_gaiadr3_Plx"])
            if row["xm_gaiadr3_Plx"] not in (None, "nan", "None", "")
            else float("nan")
        )
        eplx = (
            float(row["xm_gaiadr3_e_Plx"])
            if row["xm_gaiadr3_e_Plx"] not in (None, "nan", "None", "")
            else float("nan")
        )
        return plx / eplx if (np.isfinite(plx) and np.isfinite(eplx) and eplx > 0) else float("nan")
    except (TypeError, ValueError):
        return float("nan")


df_diaobj_catalogue["xm_gaiadr3_RPlx"] = df_diaobj_catalogue.apply(_safe_rplx, axis=1)

# ── Display summary ───────────────────────────────────────────────────────────
print(f"Base diaObject catalogue: {len(df_diaobj_catalogue)} objects")
print()
print("Objects per group:")
print(df_diaobj_catalogue["group"].value_counts().to_string())
print()
print("Objects per DDF:")
print(df_diaobj_catalogue["field"].value_counts().to_string())
print()
print("Non-null target_name:", df_diaobj_catalogue["target_name"].notna().sum())
print("Non-null Gaia DR3 name:", df_diaobj_catalogue["xm_gaiadr3_DR3Name"].notna().sum())
print()
# Dipole summary
n_dip = df_diaobj_catalogue["isDipole_any"].sum()
print(f"Objects with isDipole_any=True : {n_dip} / {len(df_diaobj_catalogue)}")
print()
display(df_diaobj_catalogue.head(5))
display(df_diaobj_catalogue[df_diaobj_catalogue["group"] == "gaia_star_stable_hq"].head(5))

## 4c. Enrich catalogue with `/api/v1/objects` — all aggregated quantities

The Fink `/api/v1/objects` endpoint returns **all aggregated statistics** for a
diaObject computed over all its diaSources.  These include:

| Category | Columns (per band `b` ∈ ugrizy) |
|----------|----------------------------------|
| Global   | `r:ra`, `r:dec`, `i:nDiaSources`, `r:firstDiaSourceMjdTai`, `r:lastDiaSourceMjdTai`, `r:target_name`, `r:observationReason` |
| psfFlux statistics | `r:{b}_psfFluxMean`, `r:{b}_psfFluxMeanErr`, `r:{b}_psfFluxSigma`, `r:{b}_psfFluxMAD`, `r:{b}_psfFluxMax`, `r:{b}_psfFluxMin`, `r:{b}_psfFluxNData` |
| psfFlux trend | `r:{b}_psfFluxLinearSlope`, `r:{b}_psfFluxLinearIntercept`, `r:{b}_psfFluxChi2` |
| psfFlux percentiles | `r:{b}_psfFluxPercentile05/25/50/75/95` |
| scienceFlux | `r:{b}_scienceFluxMean`, `r:{b}_scienceFluxMeanErr` |
| templateFlux | `r:{b}_templateFluxMean`, `r:{b}_templateFluxMeanErr` |
| apFlux | `r:{b}_apFluxMean`, `r:{b}_apFluxMeanErr`, `r:{b}_apFluxSigma` |

**Strategy**: we call the API in **batches of 50 IDs** (comma-separated)
to minimise network overhead while staying within API limits.  All columns
are fetched (`columns=None`) and merged into `df_diaobj_catalogue`.

> **Note**: `r:ra` / `r:dec` from `/objects` are the object-level coordinates
> (weighted centroid over all sources), while `r:ra` / `r:dec` in the conesearch
> are per-alert coordinates.  Both are kept under different names.

In [ ]:
# ── Fetch all /api/v1/objects data in batches ─────────────────────────────────
BATCH_SIZE = 50
SLEEP_BETWEEN_BATCHES = 0.5

all_oids = df_diaobj_catalogue["diaObjectId"].astype(str).tolist()
objects_rows = []

print(f"Fetching /api/v1/objects for {len(all_oids)} diaObjects in batches of {BATCH_SIZE}...")

for batch_start in range(0, len(all_oids), BATCH_SIZE):
    batch = all_oids[batch_start : batch_start + BATCH_SIZE]
    df_batch = fetch_objects_batch(batch)
    if not df_batch.empty:
        objects_rows.append(df_batch)
    n_done = min(batch_start + BATCH_SIZE, len(all_oids))
    if (batch_start // BATCH_SIZE) % 5 == 0:
        print(f"  {n_done}/{len(all_oids)} fetched…")
    time.sleep(SLEEP_BETWEEN_BATCHES)

if objects_rows:
    df_objects_api = pd.concat(objects_rows, ignore_index=True)
    print(f"\nAPI objects table shape : {df_objects_api.shape}")
    print(f"Columns ({len(df_objects_api.columns)}):")
    global_cols = [c for c in df_objects_api.columns if not any(c.startswith(f"i:{b}_") for b in "ugrizy")]
    print("  Global:", global_cols)
    for b in "ugrizy":
        band_cols = [c for c in df_objects_api.columns if c.startswith(f"i:{b}_")]
        if band_cols:
            print(f"  Band {b}: {band_cols}")
else:
    df_objects_api = pd.DataFrame()
    print("WARNING: /api/v1/objects returned no data — catalogue will not be enriched.")

In [ ]:
if not df_objects_api.empty:
    print("Sample row (first object):")
    display(df_objects_api.head(1).T)
else:
    print("No objects API data available.")

In [ ]:
if not df_objects_api.empty:
    obj_id_col = None
    for candidate in ("i:diaObjectId", "diaObjectId", "f:diaObjectId", "r:diaObjectId"):
        if candidate in df_objects_api.columns:
            obj_id_col = candidate
            break
    if obj_id_col is None:
        print("ERROR: cannot find diaObjectId column in /objects response.")
        print("Available columns:", df_objects_api.columns.tolist()[:20])
        df_objects_api_clean = pd.DataFrame()
    else:
        df_objects_api_clean = df_objects_api.rename(columns={obj_id_col: "diaObjectId"})
        df_objects_api_clean["diaObjectId"] = df_objects_api_clean["diaObjectId"].astype(str)

    if not df_objects_api_clean.empty:
        df_diaobj_catalogue_enriched = df_diaobj_catalogue.merge(
            df_objects_api_clean,
            on="diaObjectId",
            how="left",
            suffixes=("", "_obj"),
        )
        n_matched = df_diaobj_catalogue_enriched[df_objects_api_clean.columns[1]].notna().sum()
        print(f"Enriched catalogue shape : {df_diaobj_catalogue_enriched.shape}")
        print(f"Objects matched to /api/v1/objects: {n_matched} / {len(df_diaobj_catalogue)}")
    else:
        df_diaobj_catalogue_enriched = df_diaobj_catalogue.copy()
        print("Merge skipped — using base catalogue only.")
else:
    df_diaobj_catalogue_enriched = df_diaobj_catalogue.copy()
    print("No /api/v1/objects data — using base catalogue only.")

print(f"\nFinal catalogue columns ({len(df_diaobj_catalogue_enriched.columns)}):")
print(df_diaobj_catalogue_enriched.columns.tolist())

In [ ]:
def plot_diaobjmag_histograms_by_band(df):
    bands = ["u", "g", "r", "i", "z", "y"]
    science_columns = [f"r:{band}_scienceFluxMean" for band in bands]
    psf_columns = [f"r:{band}_psfFluxMean" for band in bands]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
    BAND_COLORS = {"u": "blue", "g": "green", "r": "red", "i": "purple", "z": "orange", "y": "brown"}
    for band, sci_col, psf_col in zip(bands, science_columns, psf_columns):
        bcolor = BAND_COLORS.get(band, "gray")
        flux_science = df[sci_col].dropna().values
        mag_science, _ = flux_to_mag(flux_science)
        axes[0].hist(
            mag_science[~np.isnan(mag_science)],
            bins=30,
            color=bcolor,
            alpha=1,
            lw=3,
            histtype="step",
            label=f"{band} (science)",
        )
        flux_psf = df[psf_col].dropna().values
        mag_psf, _ = flux_to_mag(flux_psf)
        axes[1].hist(
            mag_psf[~np.isnan(mag_psf)],
            bins=30,
            color=bcolor,
            alpha=1,
            lw=3,
            histtype="step",
            label=f"{band} (psf)",
        )
    for ax in axes:
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Count")
        ax.legend()
    axes[0].set_title("scienceFluxMean")
    axes[1].set_title("psfFluxMean")
    plt.tight_layout()
    savefig("histo_magperband")
    plt.show()

In [ ]:
def plot_diaobjmagerrmag_scatter_by_band(df):
    bands = ["u", "g", "r", "i", "z", "y"]
    science_columns = [f"r:{band}_scienceFluxMean" for band in bands]
    psf_columns = [f"r:{band}_psfFluxMean" for band in bands]
    scienceerr_columns = [f"r:{band}_scienceFluxMeanErr" for band in bands]
    psferr_columns = [f"r:{band}_psfFluxMeanErr" for band in bands]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)
    BAND_COLORS = {"u": "blue", "g": "green", "r": "red", "i": "purple", "z": "orange", "y": "brown"}
    for band, sci_col, scierr_col, psf_col, psferr_col in zip(
        bands, science_columns, scienceerr_columns, psf_columns, psferr_columns
    ):
        bcolor = BAND_COLORS.get(band, "gray")
        flux_science = df[sci_col]
        flux_scienceerr = df[scierr_col]
        mask = (~np.isnan(flux_science)) & (~np.isnan(flux_scienceerr))
        mag_science, magerr_science = flux_to_mag(flux_science[mask], flux_scienceerr[mask])
        axes[0].scatter(
            mag_science, magerr_science, color=bcolor, marker="+", lw=1, label=f"{band} (science)"
        )
        flux_psf = df[psf_col]
        flux_psferr = df[psferr_col]
        mask = (~np.isnan(flux_psf)) & (~np.isnan(flux_psferr))
        mag_psf, magerr_psf = flux_to_mag(flux_psf[mask], flux_psferr[mask])
        axes[1].scatter(mag_psf, magerr_psf, color=bcolor, marker="+", lw=1, label=f"{band} (psf)")
    for ax in axes:
        ax.set_xlabel("Magnitude")
        ax.set_ylabel("Magnitude Error")
        ax.set_ylim(0.0, 0.5)
        ax.legend()
    axes[0].set_title("scienceFluxMean")
    axes[1].set_title("psfFluxMean")
    plt.tight_layout()
    savefig("scatter_magvsmagperband")
    plt.show()

In [ ]:
plot_diaobjmag_histograms_by_band(df_diaobj_catalogue_enriched)

In [ ]:
plot_diaobjmagerrmag_scatter_by_band(df_diaobj_catalogue_enriched)

## 4d. Save enriched catalogues

Two CSV files are saved:

| File | Content |
|------|---------|
| `diaobj_catalogue.csv` | All diaObjects (all groups) with full objects API enrichment + dipole statistics |
| `diaobj_catalogue_gaia_stable.csv` | Subset: Gaia groups only |

Both files contain the per-object dipole columns:
`dipole_ndetect`, `dipole_fraction`, `isDipole_any`,
`dipoleFluxDiff_median`, `dipoleMeanFlux_median`, `dipoleLength_median`,
`dipoleAngle_median`, `dipoleChi2_median`.

In [ ]:
# ── Save full enriched catalogue ─────────────────────────────────────────────
cat_path = os.path.join(DIR_DATA, "diaobj_catalogue.csv")
df_diaobj_catalogue_enriched.to_csv(cat_path, index=False)
print(
    f"Saved: {cat_path}  ({len(df_diaobj_catalogue_enriched)} rows, {df_diaobj_catalogue_enriched.shape[1]} cols)"
)

gaia_groups = [
    "gaia_star_stable_hq",
    "gaia_star_variable",
    "gaia_brightstar_stable_unknown_parallax",
    "gaia_faintstar_stable_unknown_parallax",
    "gaia_nophotgstar_stable_unknown_parallax",
    "gaia_unknown_stable_unknown_parallax",
    "gaia_unreliable",
]


df_gaia_stable = df_diaobj_catalogue_enriched[df_diaobj_catalogue_enriched["group"].isin(gaia_groups)].copy()
print(f"Gaia-stable objects: {len(df_gaia_stable)}")

display_cols = [
    "diaObjectId",
    "group",
    "target_name",
    "xm_gaiadr3_DR3Name",
    "xm_gaiadr3_VarFlag",
    "xm_gaiadr3_RPlx",
    "ra",
    "dec",
    "field",
    "nDiaSources",
    "isDipole_any",
    "dipole_fraction",
    "dipole_ndetect",
]
for col in [
    "r:target_name",
    "r:observationReason",
    "r:firstDiaSourceMjdTai",
    "r:lastDiaSourceMjdTai",
    "r:r_psfFluxMean",
    "r:r_psfFluxSigma",
    "r:r_psfFluxLinearSlope",
    "r:r_scienceFluxMean",
    "r:r_templateFluxMean",
]:
    if col in df_gaia_stable.columns:
        display_cols.append(col)
display_cols = [c for c in display_cols if c in df_gaia_stable.columns]

gaia_cat_path = os.path.join(DIR_DATA, "diaobj_catalogue_gaia_stable.csv")
df_gaia_stable.to_csv(gaia_cat_path, index=False)
print(f"Saved: {gaia_cat_path}  ({len(df_gaia_stable)} rows, {df_gaia_stable.shape[1]} cols)")

## Set the group for calibration

In [ ]:
GROUPS_SELECTED = [
    "gaia_star_stable_hq",
    "gaia_star_variable",
    "gaia_brightstar_stable_unknown_parallax",
    "gaia_faintstar_stable_unknown_parallax",
    "gaia_nophotgstar_stable_unknown_parallax",
    "gaia_unknown_stable_unknown_parallax",
    "gaia_unreliable",
]

## 5. Download light curves for all candidates

### Columns fetched

**diaSources** (`src`) — includes all dipole columns:
| Column | Description |
|--------|-------------|
| `r:diaObjectId` | Object identifier |
| `r:diaSourceId` | Unique source detection ID |
| `r:midpointMjdTai` | Observation MJD |
| `r:psfFlux`, `r:psfFluxErr` | PSF flux and uncertainty (nJy) |
| `r:band` | Photometric band |
| `r:ra`, `r:dec` | Sky coordinates |
| `r:snr` | Signal-to-noise ratio |
| `r:visit` | Visit ID for Butler/DRP |
| `r:detector` | CCD detector number (0–188) |
| `r:x`, `r:y` | Pixel position on CCD |
| `r:xErr`, `r:yErr` | Pixel position uncertainties |
| `r:isDipole` | Dipole flag — **used to mark detections in LC plots** |
| `r:isNegative` | Negative-lobe flag |
| `r:dipoleFitAttempted` | Fit attempted flag |
| `r:dipoleFluxDiff` | Dipole flux difference (nJy) |
| `r:dipoleMeanFlux` | Dipole mean flux (nJy) |
| `r:dipoleLength` | Dipole angular separation (arcsec) |
| `r:dipoleAngle` | Dipole position angle (degrees) |
| `r:dipoleNdata` | Pixels used in fit |
| `r:dipoleChi2` | Chi² of dipole fit |

**Forced photometry** (`fp`): dipole columns are NOT available in forced photometry.

In [ ]:
# ── diaSources columns ────────────────────────────────────────────────────────
# Dipole columns added in v6 — preserved through filter_lc (not used for SNR filtering).
COLS_SRC = (
    "r:diaObjectId,r:diaSourceId,r:midpointMjdTai,"
    "r:psfFlux,r:psfFluxErr,r:band,r:ra,r:dec,r:snr,r:psfChi2,"
    "r:apFlux,r:apFluxErr,"
    # Rubin DRP: visit, detector, pixel position + uncertainties
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    # Dipole columns (v6)
    "r:isDipole,r:isNegative,r:dipoleFitAttempted,"
    "r:dipoleFluxDiff,r:dipoleFluxDiffErr,"
    "r:dipoleMeanFlux,r:dipoleMeanFluxErr,"
    "r:dipoleLength,r:dipoleAngle,r:dipoleNdata,r:dipoleChi2,"
    # Science / template fluxes
    "r:scienceFlux,r:scienceFluxErr,r:templateFlux,r:templateFluxErr"
)

# ── Forced photometry columns ─────────────────────────────────────────────────
# Dipole columns are NOT available for forced photometry.
COLS_FP = (
    "r:diaObjectId,r:midpointMjdTai,r:psfFlux,r:psfFluxErr,r:band,"
    "r:visit,r:detector,r:x,r:y,r:xErr,r:yErr,"
    "r:scienceFlux,r:scienceFluxErr"
)

lc_cache = {}  # oid → {'src': df, 'fp': df, 'group': str, 'nDiaSources': int, 'field': str}

MAX_OBJECTS = 5000
oids_sorted = sorted(all_candidates, key=lambda o: all_candidates[o]["nDiaSources"], reverse=True)
oids_to_fetch = oids_sorted[:MAX_OBJECTS]

print(f"Objects to fetch: {len(oids_to_fetch)}  (capped at {MAX_OBJECTS})")
print(
    f"nDiaSources range: {all_candidates[oids_to_fetch[0]]['nDiaSources']} "
    f"… {all_candidates[oids_to_fetch[-1]]['nDiaSources']}"
)


def _cast_rubin_cols(df: pd.DataFrame) -> pd.DataFrame:
    """Cast Rubin DRP integer columns to nullable Int64 (robust to float API returns)."""
    for col in ("r:visit", "r:detector"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    for col in ("r:x", "r:y", "r:xErr", "r:yErr"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def _cast_dipole_cols(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast dipole boolean and numeric columns to appropriate types.
    Boolean columns: r:isDipole, r:isNegative, r:dipoleFitAttempted.
    Numeric columns: all other r:dipole* columns.
    Missing columns are silently ignored.
    """
    for col in ("r:isDipole", "r:isNegative", "r:dipoleFitAttempted"):
        if col in df.columns:
            df[col] = parse_dipole_bool(df[col].fillna(False))
    for col in (
        "r:dipoleFluxDiff",
        "r:dipoleFluxDiffErr",
        "r:dipoleMeanFlux",
        "r:dipoleMeanFluxErr",
        "r:dipoleLength",
        "r:dipoleAngle",
        "r:dipoleNdata",
        "r:dipoleChi2",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


for i, oid in enumerate(oids_to_fetch):
    meta = all_candidates[oid]
    group = meta["group"]

    if group not in GROUPS_SELECTED:
        continue
    try:
        df_src = fetch_sources(oid, columns=COLS_SRC)
        df_fp = fetch_fp(oid, columns=COLS_FP)

        df_src = _cast_rubin_cols(df_src)
        df_src = _cast_dipole_cols(df_src)
        df_fp = _cast_rubin_cols(df_fp)

        df_src_f = filter_lc(df_src)
        df_fp_f = filter_lc(df_fp)

        if len(df_src_f) > NP_MIN:
            lc_cache[oid] = {
                "src": df_src_f,
                "fp": df_fp_f,
                "group": group,
                "nDiaSources": meta["nDiaSources"],
                "field": meta.get("field"),
                "target_name": meta.get("target_name"),
                # Dipole statistics from conesearch (object-level)
                "isDipole_any": meta.get("isDipole_any", False),
                "dipole_fraction": meta.get("dipole_fraction", 0.0),
                "dipole_ndetect": meta.get("dipole_ndetect", 0),
            }
            n_visits_src = df_src_f["r:visit"].dropna().nunique() if "r:visit" in df_src_f.columns else "?"
            n_visits_fp = df_fp_f["r:visit"].dropna().nunique() if "r:visit" in df_fp_f.columns else "N/A"
            # Count dipole detections in downloaded diaSources
            n_dip_src = int(df_src_f["r:isDipole"].sum()) if "r:isDipole" in df_src_f.columns else 0
            print(
                f"  [{i + 1:3d}/{len(oids_to_fetch)}] {oid}  "
                f"src={len(df_src_f):4d} (visits={n_visits_src}, dipoles={n_dip_src})  "
                f"fp={len(df_fp_f):4d} (visits={n_visits_fp})  "
                f"group={group} field={meta.get('field', '?')}"
            )
    except Exception as e:
        print(f"  [{i + 1:3d}/{len(oids_to_fetch)}] {oid}  ERROR: {e}")
    time.sleep(0.2)

print(f"\nDownloaded: {len(lc_cache)} objects")

## 6. Build per-group object lists

In [ ]:
all_groups = sorted(set(d["group"] for d in lc_cache.values()))
group_oids = {g: [] for g in all_groups}
for oid, d in lc_cache.items():
    group_oids[d["group"]].append(oid)

print("Group → object count:")
for g in sorted(group_oids, key=lambda x: -len(group_oids[x])):
    print(f"  {g:45s} {len(group_oids[g]):3d}")

CALIBRATION_EXCLUDE = {
    "solar_system",
    "tns_transient",
    "vsx_variable",
    "gcvs_variable",
    "spicy_yso",
    "blazar_x3hsp",
    "blazar_4lac",
}
CALIB_GROUPS = [g for g in all_groups if g not in CALIBRATION_EXCLUDE and not g.startswith("simbad_variable")]
print(f"\nGroups considered for calibration ({len(CALIB_GROUPS)}):")
for g in CALIB_GROUPS:
    print(f"  {g}")

## 7. Compute flatness metrics per group

In [ ]:
flatness_rows = []

for oid, data in lc_cache.items():
    group = data["group"]
    frames = [df for df in (data["fp"], data["src"]) if not df.empty]
    if not frames:
        continue
    df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
    df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
    for band in BANDS:
        df_b = df_all[df_all["r:band"] == band]
        if len(df_b) < 5:
            continue
        rms = rms_variability(df_b["r:psfFlux"].values)
        flatness_rows.append(
            {
                "group": group,
                "diaObjectId": oid,
                "band": band,
                "n_pts": len(df_b),
                "rms_var": rms,
                "mean_flux_nJy": float(df_b["r:psfFlux"].mean()),
            }
        )

df_flat = pd.DataFrame(flatness_rows)

if not df_flat.empty:
    print(f"Flatness table: {len(df_flat)} rows")
    print(df_flat.groupby("group")[["n_pts", "rms_var"]].median().round(4))
else:
    print("No flatness data — check light curve downloads.")

df_flat.to_csv(os.path.join(DIR_DATA, "flatness_metrics.csv"), index=False)
print("\nSaved flatness_metrics.csv")

## 7b. Visit summary — diaSources and Forced Photometry

In [ ]:
def build_visit_summary(lc_cache: dict, source_type: str = "src") -> pd.DataFrame:
    """
    Build a visit-level summary DataFrame from the light curve cache.
    For diaSources (source_type='src'), dipole statistics per visit are included:
      - n_dipole : number of dipole-flagged detections in this visit/band
      - isDipole_any_visit : True if any detection in this visit/band is a dipole
    """
    rows = []
    for oid, data in lc_cache.items():
        df = data.get(source_type, pd.DataFrame())
        group = data["group"]
        target_name = data["target_name"]
        if df.empty:
            continue
        if "r:visit" not in df.columns:
            continue

        group_cols = ["r:visit", "r:band"]
        if "r:detector" in df.columns:
            group_cols.insert(1, "r:detector")

        for keys, grp in df.groupby(group_cols, dropna=False):
            if not isinstance(keys, tuple):
                keys = (keys,)
            key_dict = dict(zip(group_cols, keys))

            row = {
                "diaObjectId": oid,
                "group": group,
                "target": target_name,
                "source_type": source_type,
                "visit": key_dict.get("r:visit"),
                "detector": key_dict.get("r:detector"),
                "band": key_dict.get("r:band"),
                "n_det": len(grp),
            }

            if "r:midpointMjdTai" in grp.columns:
                mjd_vals = pd.to_numeric(grp["r:midpointMjdTai"], errors="coerce")
                row["mjd_first"] = float(mjd_vals.min()) if mjd_vals.notna().any() else np.nan
            else:
                row["mjd_first"] = np.nan

            for col_in, col_out in [
                ("r:x", "x_mean"),
                ("r:y", "y_mean"),
                ("r:xErr", "xErr_mean"),
                ("r:yErr", "yErr_mean"),
            ]:
                if col_in in grp.columns:
                    vals = pd.to_numeric(grp[col_in], errors="coerce")
                    row[col_out] = float(vals.mean()) if vals.notna().any() else np.nan
                else:
                    row[col_out] = np.nan

            if "r:psfFlux" in grp.columns:
                flux = pd.to_numeric(grp["r:psfFlux"], errors="coerce")
                row["psfFlux_mean"] = float(flux.mean()) if flux.notna().any() else np.nan
            else:
                row["psfFlux_mean"] = np.nan

            # Dipole statistics per visit/band (only for diaSources)
            if source_type == "src" and "r:isDipole" in grp.columns:
                dip_mask = parse_dipole_bool(grp["r:isDipole"].fillna(False))
                row["n_dipole"] = int(dip_mask.sum())
                row["isDipole_any_visit"] = bool(dip_mask.any())
            else:
                row["n_dipole"] = 0
                row["isDipole_any_visit"] = False

            rows.append(row)

    if not rows:
        return pd.DataFrame()

    df_out = pd.DataFrame(rows)
    for col in ("visit", "detector"):
        if col in df_out.columns:
            df_out[col] = pd.to_numeric(df_out[col], errors="coerce").astype("Int64")
    df_out = df_out.sort_values(["diaObjectId", "visit", "detector", "band"]).reset_index(drop=True)
    return df_out


df_visit_summary_src = build_visit_summary(lc_cache, source_type="src")
df_visit_summary_fp = build_visit_summary(lc_cache, source_type="fp")

print("=== diaSources visit summary ===")
if not df_visit_summary_src.empty:
    print(f"  Rows             : {len(df_visit_summary_src)}")
    print(f"  Unique objects   : {df_visit_summary_src['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_visit_summary_src['visit'].dropna().nunique()}")
    print(f"  Unique detectors : {df_visit_summary_src['detector'].dropna().nunique()}")
    print(f"  Bands            : {sorted(df_visit_summary_src['band'].dropna().unique())}")
    n_dip_vis = (
        df_visit_summary_src["isDipole_any_visit"].sum()
        if "isDipole_any_visit" in df_visit_summary_src.columns
        else 0
    )
    print(f"  Visit/band rows with >=1 dipole: {n_dip_vis}")
    print(df_visit_summary_src.head(5).to_string())
else:
    print("  No diaSources visit data available.")

print("\n=== Forced Photometry visit summary ===")
if not df_visit_summary_fp.empty:
    print(f"  Rows             : {len(df_visit_summary_fp)}")
    print(f"  Unique objects   : {df_visit_summary_fp['diaObjectId'].nunique()}")
    print(f"  Unique visits    : {df_visit_summary_fp['visit'].dropna().nunique()}")
    print(df_visit_summary_fp.head(5).to_string())
else:
    print("  r:visit not available in forced photometry (expected for some API versions).")

if not df_visit_summary_src.empty:
    path_src = os.path.join(DIR_DATA, "visit_summary_src.csv")
    df_visit_summary_src.to_csv(path_src, index=False)
    print(f"\nSaved: {path_src}")

if not df_visit_summary_fp.empty:
    path_fp = os.path.join(DIR_DATA, "visit_summary_fp.csv")
    df_visit_summary_fp.to_csv(path_fp, index=False)
    print(f"Saved: {path_fp}")

## 8. Plot: flatness distribution per group

In [ ]:
if df_flat.empty:
    print("No data.")
else:
    groups_present = [g for g in all_groups if g in df_flat["group"].unique()]
    bands_present = [b for b in BANDS if b in df_flat["band"].unique()]
    short_labels = [g.replace("_", "\n") for g in groups_present]

    fig, axes = plt.subplots(1, len(bands_present), figsize=(3.2 * len(bands_present), 5), sharey=True)
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = df_flat[df_flat["band"] == band]
        data_per_group = [df_b[df_b["group"] == g]["rms_var"].dropna().values for g in groups_present]
        bp = ax.boxplot(data_per_group, labels=short_labels, patch_artist=True, notch=False, showfliers=True)
        for patch in bp["boxes"]:
            patch.set_facecolor(BAND_COLORS.get(band, "#aaa"))
            patch.set_alpha(0.5)
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.tick_params(axis="x", rotation=60, labelsize=7)
        ax.set_yscale("log")
        ax.axhline(0.05, ls="--", color="green", lw=0.8, alpha=0.6)

    axes[0].set_ylabel("Normalised RMS  σ/<f>")
    fig.suptitle("Flux variability by source class — lower = flatter", fontsize=12, fontweight="bold", y=1.02)
    plt.tight_layout()
    savefig("01_flatness_boxplot_by_group")
    plt.show()

## 9. Plot light curves — top-NC objects per group

Only groups in `CALIB_GROUPS` are plotted (transients and confirmed variables excluded  
from the calibration ranking, but still saved in all data files).

**Dipole marker**: detections flagged as `r:isDipole == True` in the diaSources are
overlaid as large **light-grey circles** (○) on the psfFlux vs time plots.
They are drawn on top of the standard band-colour errorbar markers so that
dipole contamination is immediately visible.

In [ ]:
def rank_oids(oid_list, nc=NC):
    scored = [(o, len(lc_cache[o]["fp"]) + len(lc_cache[o]["src"])) for o in oid_list if o in lc_cache]
    return [o for o, _ in sorted(scored, key=lambda x: -x[1])[:nc]]


def getTminTmax(df, df_src, df_fp):
    t = df["r:midpointMjdTai"].values
    t_src = df_src["r:midpointMjdTai"].values
    t_fp = df_fp["r:midpointMjdTai"].values
    if len(t_src) > 0:
        tmin = t_src.min()
    else:
        tmin = t.min()
    tmax = t.max()
    return tmin, tmax


def getYminYmax(df, df_src, df_fp):
    y = df["r:psfFlux"].values
    y_src = df_src["r:psfFlux"].values
    y_fp = df_fp["r:psfFlux"].values
    if len(y_src) > 0:
        ymin = y_src.min()
        ymax = y_src.max()
    else:
        ymin = y.min()
        ymax = y.max()
    magmin, _ = flux_to_mag(ymax)
    magmax, _ = flux_to_mag(ymin)
    return ymin, ymax, magmin, magmax


def plot_lc_grid(oid_list, group, mode="flux", nc=NC):
    """
    Plot a grid of light curves (one row per object, one column per band).

    Dipole overlay (mode='flux' only):
      Detections flagged r:isDipole == True in the diaSources DataFrame are
      overlaid as large light-grey open circles (marker='o', color='lightgrey',
      zorder=5) at the corresponding (Δt, psfFlux) positions.  This makes
      dipole-contaminated epochs immediately visible in each light curve panel.
      Forced-photometry points are never marked as dipoles (column absent).
    """
    top = rank_oids(oid_list, nc)
    n_obj = len(top)
    if n_obj == 0:
        print(f"  No objects for group {group}.")
        return

    fig, axes = plt.subplots(
        n_obj, len(BANDS), figsize=(2.8 * len(BANDS), 2.6 * n_obj), sharex=False, sharey=False, squeeze=False
    )

    for row_idx, oid in enumerate(top):
        d = lc_cache[oid]
        frames = [df for df in (d.get("fp", pd.DataFrame()), d.get("src", pd.DataFrame())) if not df.empty]
        if not frames:
            continue
        df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_all = df_all.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        df_fp = d["fp"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_fp = df_fp.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(drop=True)
        df_src = d["src"].drop_duplicates(subset=["r:midpointMjdTai", "r:band"])
        df_src = df_src.dropna(subset=["r:midpointMjdTai", "r:psfFlux", "r:psfFluxErr"]).reset_index(
            drop=True
        )

        tmin, tmax = getTminTmax(df_all, df_src, df_fp)
        ymin, ymax, magmin, magmax = getYminYmax(df_all, df_src, df_fp)
        deltamag = np.max([magmax - magmin, 3.0])
        centermag = (magmax + magmin) / 2.0
        ymagmin = centermag - deltamag / 2.0
        ymagmax = centermag + deltamag / 2.0

        first_band = -1
        ax_first_band = None
        for col_idx, band in enumerate(BANDS):
            ax = axes[row_idx][col_idx]
            df_b = df_all[df_all["r:band"] == band].sort_values("r:midpointMjdTai")
            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            if first_band == -1:
                first_band = col_idx
                ax_first_band = ax

            if mode == "flux":
                mask = np.isfinite(df_b["r:psfFlux"].values) & np.isfinite(df_b["r:psfFluxErr"].values)
            else:
                mask = np.isfinite(df_b["mag"].values) & np.isfinite(df_b["mag_err"].values)
            df_b = df_b[mask].reset_index(drop=True)
            if len(df_b) < 3:
                ax.set_visible(False)
                continue

            t = df_b["r:midpointMjdTai"].values
            dt = t - tmin
            dtmin = dt.min()
            dtmax = dt.max()

            color = BAND_COLORS.get(band, "gray")
            if mode == "flux":
                y, yerr = df_b["r:psfFlux"].values, df_b["r:psfFluxErr"].values
            else:
                y, yerr = df_b["mag"].values, df_b["mag_err"].values
                ax.invert_yaxis()
            ax.errorbar(dt, y, yerr=yerr, fmt="o", ms=3, lw=0.8, elinewidth=0.8, color=color, alpha=0.8)

            # ── Dipole overlay (flux mode only) ───────────────────────────────
            # Mark detections flagged as r:isDipole with a large light-grey
            # open circle drawn on top of the band-colour markers.
            if mode == "flux" and "r:isDipole" in df_b.columns:
                dip_mask = parse_dipole_bool(df_b["r:isDipole"].fillna(False))
                if dip_mask.any():
                    dt_dip = dt[dip_mask.values]
                    y_dip = y[dip_mask.values]
                    ax.scatter(
                        dt_dip,
                        y_dip,
                        marker="o",
                        s=120,
                        facecolors="none",
                        edgecolors="lightgrey",
                        linewidths=1.5,
                        zorder=5,
                        alpha=0.9,
                        label="dipole" if col_idx == first_band else None,
                    )

            rms = rms_variability(df_b["r:psfFlux"].values)
            # Count dipoles in this band panel
            n_dip_b = 0
            if "r:isDipole" in df_b.columns:
                n_dip_b = int(parse_dipole_bool(df_b["r:isDipole"].fillna(False)).sum())
            dip_label = f" dip={n_dip_b}" if n_dip_b > 0 else ""
            ax.set_title(f"{band} N={len(df_b)} rms={rms:.3f}{dip_label}", fontsize=7, pad=2, color=color)
            ax.set_xlabel("Δt (days)", fontsize=7)
            ax.tick_params(labelsize=7)
            ax.set_xlim(-1.0, dtmax + 1.0)
            if mode == "flux":
                ax.set_ylim(0.0, 1.2 * ymax)
            else:
                ax.set_ylim(ymagmax, ymagmin)

        if ax_first_band is not None:
            deep_field = lc_cache[oid].get("field")
            field_label = f"  [{deep_field}]" if deep_field else ""
            ax_first_band.set_ylabel(
                f"{oid}{field_label}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=10
            )
        else:
            axes[row_idx][0].set_ylabel(f"{oid}\n{'flux (nJy)' if mode == 'flux' else 'AB mag'}", fontsize=10)

    fig.suptitle(f"Group: {group}  |  mode={mode}", fontsize=11, fontweight="bold", y=1.01)
    plt.tight_layout()
    safe = group.replace("/", "_").replace(" ", "_")
    savefig(f"02_lc_{safe}_{mode}")
    plt.show()


print("Plot functions defined.")

In [ ]:
# Plot calibration-candidate groups only
groups_to_plot = [g for g in CALIB_GROUPS if len(group_oids.get(g, [])) >= 2]

for group in groups_to_plot:
    print(f"\n=== {group} ({len(group_oids[group])} objects) ===")
    plot_lc_grid(group_oids[group], group, mode="flux")

In [ ]:
for group in groups_to_plot:
    print(f"\n=== {group} (magnitude) ===")
    plot_lc_grid(group_oids[group], group, mode="mag")

## 10. Calibration summary scatter plot

In [ ]:
if df_flat.empty:
    print("No data.")
else:
    summary = (
        df_flat.groupby(["group", "band"])
        .agg(
            median_rms=("rms_var", "median"),
            mean_flux=("mean_flux_nJy", "mean"),
            n_obj=("diaObjectId", "nunique"),
            n_pts=("n_pts", "sum"),
        )
        .reset_index()
    )
    bands_present = [b for b in BANDS if b in summary["band"].unique()]
    fig, axes = plt.subplots(1, len(bands_present), figsize=(4.5 * len(bands_present), 5))
    if len(bands_present) == 1:
        axes = [axes]

    for ax, band in zip(axes, bands_present):
        df_b = summary[summary["band"] == band]
        ax.scatter(
            df_b["mean_flux"],
            df_b["median_rms"],
            s=80 * np.sqrt(df_b["n_pts"] / max(df_b["n_pts"].max(), 1) + 0.1),
            c=BAND_COLORS.get(band, "gray"),
            alpha=0.8,
            edgecolors="k",
            linewidths=0.5,
        )
        for _, row in df_b.iterrows():
            ax.annotate(
                row["group"],
                (row["mean_flux"], row["median_rms"]),
                fontsize=6,
                ha="left",
                va="bottom",
                xytext=(3, 2),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("Mean flux (nJy)")
        ax.set_ylabel("Median σ/<f>")
        ax.set_title(f"Band {band}", color=BAND_COLORS.get(band, "k"), fontweight="bold")
        ax.axhline(0.05, ls="--", color="green", lw=1, alpha=0.6, label="5%")
        ax.legend(fontsize=7)

    fig.suptitle(
        "Calibration suitability  (best = bright & flat = bottom-right)",
        fontsize=11,
        fontweight="bold",
        y=1.02,
    )
    plt.tight_layout()
    savefig("03_calibration_summary")
    plt.show()

    print("\nFull summary table:")
    display(summary.sort_values(["band", "median_rms"]).head(40))

## 11. Save all light curves to Parquet

All groups are saved — including transients and variables — so that `04_fink_sselectDIAObject_tovisitIddetector.ipynb`
can select any object regardless of its classification.

**`*_src.parquet`** files contain all dipole columns (`r:isDipole`, `r:dipoleFluxDiff`, etc.)
in addition to the standard photometric and DRP columns.

In [ ]:
for group in all_groups:
    oids = group_oids[group]
    all_fp, all_src = [], []
    for oid in oids:
        d = lc_cache.get(oid, {})
        for df, store in ((d.get("fp", pd.DataFrame()), all_fp), (d.get("src", pd.DataFrame()), all_src)):
            if not df.empty:
                tmp = df.copy()
                tmp["diaObjectId"] = oid
                tmp["group"] = group
                tmp["target"] = d["target_name"]
                tmp["field"] = d["field"]
                store.append(tmp)
    safe = group.replace("/", "_")
    for store, tag in ((all_fp, "fp"), (all_src, "src")):
        if store:
            path = os.path.join(DIR_DATA, f"{safe}_{tag}.parquet")
            pd.concat(store, ignore_index=True).to_parquet(path, index=False)
            print(f"  Saved {path}")

print("\nAll data saved.")

## 12. Build and save the global visit index

In [ ]:
if not df_visit_summary_src.empty:
    idx_path = os.path.join(DIR_DATA, "visit_index.csv")
    df_visit_summary_src.to_csv(idx_path, index=False)
    print(
        f"visit_index.csv saved ({len(df_visit_summary_src)} rows, "
        f"{df_visit_summary_src['visit'].dropna().nunique()} unique visits)"
    )

if not df_visit_summary_fp.empty:
    idx_fp_path = os.path.join(DIR_DATA, "visit_index_fp.csv")
    df_visit_summary_fp.to_csv(idx_fp_path, index=False)
    print(
        f"visit_index_fp.csv saved ({len(df_visit_summary_fp)} rows, "
        f"{df_visit_summary_fp['visit'].dropna().nunique()} unique visits)"
    )

## 13. Final ranking — recommended source class for calibration

In [ ]:
if df_flat.empty:
    print("No flatness data.")
else:
    df_calib = df_flat[df_flat["group"].isin(CALIB_GROUPS)]
    ranking = (
        df_calib.groupby("group")
        .agg(
            n_objects=("diaObjectId", "nunique"),
            n_points=("n_pts", "sum"),
            median_rms=("rms_var", "median"),
            mean_flux_nJy=("mean_flux_nJy", "mean"),
        )
        .sort_values("median_rms")
        .reset_index()
    )
    ranking["mean_mag"] = -2.5 * np.log10(ranking["mean_flux_nJy"] / AB_FLUX_ZERO)

    print("Ranking by photometric flatness (ascending RMS) — calibration groups only:")
    print("=" * 90)
    print(
        ranking[["group", "n_objects", "n_points", "median_rms", "mean_mag"]].to_string(
            index=False, float_format="{:.4f}".format
        )
    )

    best = ranking.iloc[0]
    print(f"\n==> Best class for atmospheric transparency calibration:")
    print(
        f"    {best['group']}  (median σ/<f> = {best['median_rms']:.4f},  mean mag ≈ {best['mean_mag']:.2f})"
    )
    print(
        f"\nNote: gaia_star_stable_hq (Plx/ePlx >= {GAIA_RPLX_MIN}) is the purest "
        f"geometric-parallax subsample if available."
    )